# Módulo 08: Regresión logística y decisiones de clasificación

[Abrir en Colab](https://colab.research.google.com/github/sgevatschnaider/data-science-business-decisions/blob/main/notebooks/08-regresion-logistica.ipynb)

**Pregunta de decisión:** ¿A quién conviene asignar una acción cuando los errores tienen costos distintos y la capacidad es limitada?

**Autor:** Sergio Gevatschnaider


## Objetivos

- Interpretar probabilidad, logit y odds.
- Leer matrices de confusión y métricas por clase.
- Distinguir discriminación, calibración y decisión.
- Elegir umbrales con costos y restricciones de capacidad.

**Criterio de éxito:** el resultado debe cambiar o sostener una acción concreta, superar una referencia y declarar límites.


## 1. Entorno reproducible

Registramos versiones y semilla antes de producir evidencia. Ejecutá siempre **Runtime → Run all** en Colab.


In [ ]:
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
import sklearn

SEED = 42
np.random.seed(SEED)
print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})

## 2. Experimento base

El bloque siguiente construye una referencia mínima y verificable. No representa todavía la recomendación final.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=600, weights=[0.8, 0.2], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]
rows = []
for threshold in [0.2, 0.4, 0.6, 0.8]:
    pred = (prob >= threshold).astype(int)
    rows.append({
        "umbral": threshold,
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred),
        "positivos_seleccionados": pred.sum(),
    })
display(pd.DataFrame(rows))
print({"ROC_AUC_test": roc_auc_score(y_test, prob), "Brier_test": brier_score_loss(y_test, prob)})

## 3. Evidencia visual

Una visualización útil permite comparar, muestra unidades y deja visible la incertidumbre o variación relevante.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
CalibrationDisplay.from_predictions(y_test, prob, n_bins=8, ax=axes[0])
axes[0].set_title("Confiabilidad de probabilidades")
PrecisionRecallDisplay.from_predictions(y_test, prob, ax=axes[1])
axes[1].set_title("Ranking con clase minoritaria")
plt.tight_layout()

## 4. Comparación para decidir

Una campaña de retención solo puede contactar al 15 por ciento de la cartera; el equipo debe ordenar riesgo y estimar valor neto por contacto.

La tabla fuerza una comparación entre alternativas, costos o criterios. Adaptala a las unidades del caso.


In [ ]:
pd.DataFrame(rows).assign(valor_estimado=lambda d: d.recall * 30 - (1 - d.precision.fillna(0)) * 4).sort_values('valor_estimado', ascending=False)

## 5. Desafío de transferencia

**Elegí una política de contacto con capacidad limitada y costos distintos por falso positivo y falso negativo.**

1. Definir clase positiva y consecuencias de error.
2. Construir un baseline de prevalencia.
3. Evaluar ranking, calibración y métricas a varios umbrales.
4. Elegir una política compatible con capacidad y valor.

Antes de continuar, escribí una hipótesis, una condición que la refutaría y el costo de una decisión equivocada.

### Registro de decisión

Completá la celda siguiente como evidencia de cierre del laboratorio.


In [ ]:
decision_record = {
    "pregunta": '¿A quién conviene asignar una acción cuando los errores tienen costos distintos y la capacidad es limitada?',
    "hipotesis": "Completar antes del análisis",
    "evidencia": "Registrar la tabla o visualización que cambia la decisión",
    "recomendacion": "Expresar acción, población y horizonte",
    "limitacion": "Indicar qué podría invalidar la conclusión",
    "responsable": "Asignar dueño y fecha de revisión",
}
pd.Series(decision_record, name="registro_de_decision")

## 6. Cierre verificable

**Entregable:** Modelo probabilístico con curva de calibración, matriz de confusión, selección de umbral y matriz de costos.

- Hallazgo principal:
- Evidencia que lo respalda:
- Comparación contra baseline o escenario alternativo:
- Limitación:
- Acción, responsable y fecha de revisión:

Material elaborado por el profesor Sergio Gevatschnaider.
